In [0]:
"""
02_silver_transformation.py

Purpose:
- Standardize column names
- Cast proper data types
- Flatten structures if needed
- Create cleaned Silver tables
"""

import re
from pyspark.sql.functions import (
    col,
    to_timestamp,
    year
)

# --------------------------------------------------
# Read Bronze Tables
# --------------------------------------------------

item_df = spark.table("bronze_item")

event_df = spark.table("bronze_event")

In [0]:
# --------------------------------------------------
# Helper Functions
# --------------------------------------------------

def to_snake_case(name):
    name = re.sub(r'([a-z0-9])([A-Z])', r'\\1_\\2', name)
    return name.lower()

def standardize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))
    return df

In [0]:
# --------------------------------------------------
# Standardize Column Names
# --------------------------------------------------

item_df = standardize_columns(item_df)

# event_df = standardize_columns(event_df)

event_df.select('event_payload').show(truncate=False)
# --------------------------------------------------
# Explore Schema
# --------------------------------------------------

print("Item Schema")
item_df.printSchema()

print("Event Schema")
event_df.printSchema()

event_df.select('event_payload').show(truncate=False)

In [0]:
# --------------------------------------------------
# Cast Data Types
# --------------------------------------------------
# NOTE:
# Update the timestamp column name if needed
# based on actual dataset structure.
event_df = (
    event_df
    .withColumn(
        "event_time",
        to_timestamp(col("event_time"))
    )
)
# --------------------------------------------------
# Add Year Column
# --------------------------------------------------

event_df = event_df.withColumn(
    "year",
    year(col("event_time"))
)
# --------------------------------------------------
# Flatten Structs (Example)
# --------------------------------------------------
#Clean Escaped Quotes from struct type event_payload
from pyspark.sql.functions import regexp_replace

# event_df = event_df.withColumn(
#     "event_payload_clean",
#     regexp_replace(
#         col("event_payload"),
#         '""',
#         '"'
#     )
# )
# #Remove Outer Quotes
# from pyspark.sql.functions import expr

# event_df = event_df.withColumn(
#     "event_payload_clean",
#     expr("substring(event_payload_clean, 2, length(event_payload_clean)-2)")
# )

# event_df.show()
# event_df.select("event_payload").show(truncate=False)
# display(event_df.select("event_payload"))


#event_df

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

#Define Schema
payload_schema = StructType([
    StructField(
        "event_name",
        StringType(),
        True
    ),
    StructField(
        "platform",
        StringType(),
        True
    ),
    StructField(
        "parameter_name",
        StringType(),
        True
    ),
    StructField(
        "parameter_value",
        StringType(),
        True
    )
])

# #Parse JSON
# from pyspark.sql.functions import from_json

event_df = event_df.withColumn(
    "payload_json",
    from_json(
        col("event_payload"),
        payload_schema
    )
)

event_df.select("payload_json.*").show(truncate=False)

#Flatten Fields
# event_df = event_df.select(
#     "*",

#     col("payload_struct.event_name").alias("event_type"),

#     col("payload_struct.platform").alias("platform"),
#     col("payload_struct.parameter_name").alias("parameter_name"),

#     col("payload_struct.parameter_value").alias("item_id")

# )

# event_df.show()

In [0]:
# # --------------------------------------------------
# # Save Silver Tables
# # --------------------------------------------------

item_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_item")

event_df.write.format("delta") \
    .partitionBy("year") \
    .mode("overwrite") \
    .saveAsTable("silver_event")

# print("Silver layer completed successfully.")